# Hangman RNN: inference examples

Run from the `notebooks/` directory. Win-rate benchmarks against the self-hosted game engine are in `03_evaluation.ipynb`.

#### 1. Load model

In [ ]:
import sys
sys.path.insert(0, "..")  # make the `hangman` package importable

import torch
from hangman.models import CustomNN

model = CustomNN(chars_len=26, embed_dim=16, hidden_dim=128, num_layers=2, dropout=0.2)
model.load_state_dict(torch.load("../models/model_rnn.pth", map_location=torch.device("cpu")))
model.eval()

#### 2. Load inference methodology

In [ ]:
from hangman import HangmanBasic, HangmanRNN

In [19]:
## Inference
def guess(model, masked_word, guessed_letters):
    stripped = masked_word.replace(" ", "")
    if all(c == '_' for c in stripped):
        # First guess: prioritize vowels not yet guessed
        vowels = ['e', 'a', 'o', 'i', 'u']
        for v in vowels:
            if v not in guessed_letters:
                return v, None, None
        # If all vowels are wrong, fall back to consonants (ETAOIN SHRDLU consonants)
        sorted_consonants = ['t', 'n', 's', 'h', 'r', 'd', 'l', 'b', 'c', 'f', 'g', 'j', 'k', 'm', 'p', 'q', 'v', 'w', 'x', 'y', 'z']
        for c in sorted_consonants:
            if c not in guessed_letters:
                return c, None, None

    word_len = len(stripped)
    x_input = encode_word(stripped)

    model.eval()
    with torch.no_grad():
        lengths = torch.tensor([word_len])  # batch of 1
        logits = model(x_input.unsqueeze(0), lengths)
        probs = torch.softmax(logits[0], dim=-1)    # max_word_len x all_chars_len
        
        # 1. Zero out positions already known (non-zero in input)
        known_positions_mask = x_input.sum(dim=1) > 0  # [T]
        probs[known_positions_mask] = 0.0

        # 2. Zero out previously guessed letters
        if guessed_letters:
            guessed_char_idx = torch.tensor([char_to_idx.get(x) for x in guessed_letters])
            unknown_positions_mask = torch.tensor([i for i in range(word_len) if not known_positions_mask[i]])
            probs[unknown_positions_mask[:, None], guessed_char_idx] = 0

        # 3. normalize probabilities within each position
        row_sums = probs.sum(dim=1, keepdim=True) + 1e-8  # avoid division by zero
        probs_normalized = probs / row_sums

        # 4. pick max probability among all positions and characters
        guessed_pos, char = torch.where(probs_normalized == probs_normalized.max())
        guessed_char = idx_to_char[char[0].item()]

    return guessed_char, guessed_pos, probs_normalized

#### 3. Test model

In [20]:
## Example 1
masked_word = 'app_e'
hangman_input = ' '.join(masked_word)
guessed_letters = ['a', 'p', 'e', 'b', 'c', 'd', 'h']


char, pos, probs = guess(model, hangman_input, guessed_letters)

if len(set(masked_word)) == 1:
    print(f"Next guess: '{char}'")
else:
    print(f"Next guess: '{char}' at position {pos}")
    print("Probability matrix (masked positions only):")
    masked_positions = [idx for idx, char in enumerate(masked_word) if char == '_']
    for i in masked_positions:
        probs_dict = {idx_to_char[j]: float(probs[i, j]) for j in range(all_chars_len) if probs[i,j]>0}
        print(f"Position {i}: ", probs_dict)
        print(sorted(probs_dict, key=probs_dict.get, reverse=True))

Next guess: 'l' at position tensor([3])
Probability matrix (masked positions only):
Position 3:  {'f': 0.00012836948735639453, 'g': 0.001108763855881989, 'i': 0.19128884375095367, 'j': 0.0002790417929645628, 'k': 0.00015477229317184538, 'l': 0.6618263721466064, 'm': 0.00042593933176249266, 'n': 0.001427668146789074, 'o': 0.023885030299425125, 'q': 2.3676307137066033e-06, 'r': 0.05030466243624687, 's': 0.029816830530762672, 't': 0.00936949159950018, 'u': 0.024286340922117233, 'v': 0.0003797242825385183, 'w': 0.00012386047455947846, 'x': 1.4550921150657814e-05, 'y': 0.005062872543931007, 'z': 0.00011447598080849275}
['l', 'i', 'r', 's', 'u', 'o', 't', 'y', 'n', 'g', 'm', 'v', 'j', 'k', 'f', 'w', 'z', 'x', 'q']


In [21]:
## Example 2
masked_word = '____'
hangman_input = ' '.join(masked_word)
guessed_letters = []


char, pos, probs = guess(model, hangman_input, guessed_letters)

if len(set(masked_word)) == 1:
    print(f"Next guess: '{char}'")
else:
    print(f"Next guess: '{char}' at position {pos}")
    print("Probability matrix (masked positions only):")
    masked_positions = [idx for idx, char in enumerate(masked_word) if char == '_']
    for i in masked_positions:
        probs_dict = {idx_to_char[j]: float(probs[i, j]) for j in range(all_chars_len) if probs[i,j]>0}
        print(f"Position {i}: ", probs_dict)
        print(sorted(probs_dict, key=probs_dict.get, reverse=True))

Next guess: 'e'


#### 4. Results
On the original external evaluation (1,000 recorded games on unseen words) this model reached an overall success rate of **0.517**. Reproducible local benchmarks are in `03_evaluation.ipynb`.